<a href="https://colab.research.google.com/github/AbdullahIdrees291/flyrank-ml-internship/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdullahIdrees291/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


## 1. Unit of analysis + time window

One row = one client × content × report_date (one content item's daily performance for a client). Time window = March 1–31, 2026.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
print(HF_TOKEN is not None)
print(HF_TOKEN[:8] if HF_TOKEN else "NO TOKEN")

True
hf_ZfkAk


In [ ]:
import duckdb

con = duckdb.connect()

con.execute("""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("Hugging Face secret set")

Hugging Face secret set


In [ ]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("HF authentication configured")

HF authentication configured


In [ ]:
con.sql("""
SELECT name, type, provider
FROM duckdb_secrets()
""").show()

┌──────────┬─────────────┬──────────┐
│   name   │    type     │ provider │
│ varchar  │   varchar   │ varchar  │
├──────────┼─────────────┼──────────┤
│ hf_token │ huggingface │ config   │
└──────────┴─────────────┴──────────┘



In [ ]:
rel = "hf://datasets/FlyRank/internship-warehouse"

result = con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""")

result.show()

┌─────────┬────────────┬────────────┐
│  rows   │  min_date  │  max_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘



## 2. Fields: feature / label / context / excluded

Features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- scroll_events

Label / proxy:
- Future search-performance decline, based on future GSC clicks.

Context:
- report_date
- client_hash_id
- content_hash_id
- month

Excluded:
- gsc_sum_position — raw sum is not directly used because average position is the meaningful position metric.
- ga4_data_available / gsc_data_available — used only to filter/check data availability, not as model features.
- ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other — excluded from this first five-feature frame to keep the slice focused on core search performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
con.sql("""
DESCRIBE SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [ ]:
cols = con.sql("""
DESCRIBE SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").fetchall()

for i, col in enumerate(cols, 1):
    print(i, col[0])

1 report_date
2 client_hash_id
3 content_hash_id
4 client_has_gsc
5 client_has_ga4
6 gsc_data_available
7 ga4_data_available
8 gsc_impressions
9 gsc_clicks
10 gsc_sum_position
11 gsc_avg_position
12 ga4_pageviews
13 ga4_sessions
14 ga4_users
15 ga4_engaged_sessions
16 ga4_total_engagement_sec
17 sessions_organic
18 sessions_direct
19 sessions_referral
20 sessions_social
21 sessions_paid
22 sessions_ai
23 ai_chatgpt
24 ai_perplexity
25 ai_gemini
26 ai_copilot
27 ai_claude
28 ai_meta
29 ai_other
30 scroll_events
31 month


In [ ]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘



In [ ]:
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS c
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



In [ ]:
con.sql("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").show()

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



In [ ]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""").show()

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘



In [ ]:
feature_df = con.sql("""
WITH x AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        scroll_events,
        LEAD(gsc_clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id
            ORDER BY report_date
        ) AS next_day_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
)
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    scroll_events,
    CASE
        WHEN next_day_clicks IS NOT NULL
         AND next_day_clicks < gsc_clicks
        THEN 1 ELSE 0
    END AS label
FROM x
WHERE next_day_clicks IS NOT NULL
""").df()

feature_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events,label
0,2026-03-25,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,7,0,4.857143,<NA>,<NA>,0
1,2026-03-26,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,35,0,8.171429,<NA>,<NA>,0
2,2026-03-27,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,37,0,7.675676,<NA>,<NA>,0
3,2026-03-28,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,43,0,7.465116,<NA>,<NA>,0
4,2026-03-29,client_0797ff3a1fc9a6a5,content_c88d7630a340a086,46,0,7.347826,<NA>,<NA>,0


Available when?
- gsc_impressions — decision moment se pehle recorded search impressions available hoti hain.
- gsc_clicks — decision moment se pehle recorded search clicks available hoti hain.
- gsc_avg_position — decision moment se pehle recorded average search position available hoti hai.
- ga4_sessions — decision moment se pehle recorded sessions available hoti hain, jab GA4 data available ho.
- scroll_events — decision moment se pehle recorded scroll events available hote hain.

## 3. Verify it with queries (grain, counts, missing values, windows)

Leakage experiment: I intentionally added a label-derived column (leaky_label). The quick score became artificially perfect because the feature directly contained the answer. I removed leaky_label, so the final feature frame contains only information available at the decision moment.

In [ ]:
feature_df["leaky_label"] = feature_df["label"]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = feature_df[["leaky_label"]]
y = feature_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

score = accuracy_score(y_test, model.predict(X_test))
print("Leaky score:", score)

Leaky score: 1.0


In [ ]:
feature_df = feature_df.drop(columns=["leaky_label"])

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

Limitation: History depth differs across clients, and GA4 data is unavailable for many rows. Therefore, this March slice is not equally complete for every client, so the results are directional rather than universal.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.